# 05 - Baseline Models

Establish performance baselines using traditional ML models before transformer development.

## Objectives
- Train XGBoost/Random Forest on integrated dataset
- Establish benchmark metrics for comparison
- Analyze feature importance
- Identify challenging predictions

## Models
- Random Forest
- XGBoost
- Gradient Boosting

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb

# Configuration
plt.style.use('seaborn-v0_8-whitegrid')

DATA_DIR = Path('../../data/processed')
RESULTS_DIR = Path('../../results/model_experiments')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Training Data

In [ ]:
# Load processed data
train_path = DATA_DIR / 'train.parquet'
val_path = DATA_DIR / 'validation.parquet'

if train_path.exists():
    train_df = pd.read_parquet(train_path)
    val_df = pd.read_parquet(val_path)
    print(f"Train: {len(train_df):,} samples")
    print(f"Validation: {len(val_df):,} samples")
else:
    print("Run notebook 04_data_integration.ipynb first")
    train_df = None

In [ ]:
# Define targets and features
TARGETS = ['SOC', 'pH', 'clay', 'sand', 'N', 'CEC']
EXCLUDE_COLS = ['profile_id', 'latitude', 'longitude', 'date', 
                'weather_sequence', 'spatial_group', 'mask', 
                'original_targets', 'masked_targets'] + TARGETS

if train_df is not None:
    available_targets = [t for t in TARGETS if t in train_df.columns]
    feature_cols = [c for c in train_df.columns if c not in EXCLUDE_COLS]
    
    print(f"Targets: {available_targets}")
    print(f"Features: {len(feature_cols)}")

## 2. Model Configuration

Use settings typical of a production RF/GBM/XGB trainer.

In [ ]:
# Model configurations (from production)
MODEL_CONFIGS = {
    'RandomForest': {
        'model': RandomForestRegressor,
        'params': {
            'n_estimators': 100,
            'max_depth': 15,
            'min_samples_split': 10,
            'min_samples_leaf': 4,
            'random_state': 42,
            'n_jobs': -1
        },
        'weight': 0.40
    },
    'GradientBoosting': {
        'model': GradientBoostingRegressor,
        'params': {
            'n_estimators': 100,
            'max_depth': 5,
            'learning_rate': 0.05,
            'subsample': 0.8,
            'random_state': 42
        },
        'weight': 0.35
    },
    'XGBoost': {
        'model': xgb.XGBRegressor,
        'params': {
            'n_estimators': 100,
            'max_depth': 6,
            'learning_rate': 0.1,
            'min_child_weight': 3,
            'subsample': 0.8,
            'random_state': 42
        },
        'weight': 0.25
    }
}

print("Model configurations loaded")
for name, config in MODEL_CONFIGS.items():
    print(f"  {name}: weight={config['weight']}")

## 3. Train and Evaluate Models

In [ ]:
def train_and_evaluate(X_train, y_train, X_val, y_val, groups, model_config):
    """Train model and return metrics."""
    model = model_config['model'](**model_config['params'])
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    
    # Metrics
    metrics = {
        'train_r2': r2_score(y_train, train_pred),
        'train_rmse': np.sqrt(mean_squared_error(y_train, train_pred)),
        'val_r2': r2_score(y_val, val_pred),
        'val_rmse': np.sqrt(mean_squared_error(y_val, val_pred)),
        'val_mae': mean_absolute_error(y_val, val_pred)
    }
    
    # Cross-validation with spatial groups
    if groups is not None:
        gkf = GroupKFold(n_splits=5)
        cv_scores = cross_val_score(
            model_config['model'](**model_config['params']),
            X_train, y_train, groups=groups,
            cv=gkf, scoring='r2'
        )
        metrics['cv_r2_mean'] = cv_scores.mean()
        metrics['cv_r2_std'] = cv_scores.std()
    
    return model, metrics

In [ ]:
# Train all models for all targets
results = []
models = {}

if train_df is not None:
    X_train = train_df[feature_cols].fillna(0)
    X_val = val_df[feature_cols].fillna(0)
    groups = train_df['spatial_group'] if 'spatial_group' in train_df.columns else None
    
    for target in available_targets:
        print(f"\n{'='*50}")
        print(f"Training for: {target}")
        print('='*50)
        
        y_train = train_df[target]
        y_val = val_df[target]
        
        # Skip if too many missing
        if y_train.isna().mean() > 0.5:
            print(f"  Skipping {target}: >50% missing")
            continue
        
        # Remove missing
        train_mask = ~y_train.isna()
        val_mask = ~y_val.isna()
        
        for model_name, model_config in MODEL_CONFIGS.items():
            model, metrics = train_and_evaluate(
                X_train[train_mask], y_train[train_mask],
                X_val[val_mask], y_val[val_mask],
                groups[train_mask] if groups is not None else None,
                model_config
            )
            
            print(f"  {model_name}: Val R²={metrics['val_r2']:.3f}, RMSE={metrics['val_rmse']:.3f}")
            
            results.append({
                'target': target,
                'model': model_name,
                **metrics
            })
            models[(target, model_name)] = model

## 4. Results Summary

In [ ]:
# Create results DataFrame
if results:
    results_df = pd.DataFrame(results)
    
    # Pivot for comparison
    pivot = results_df.pivot_table(
        index='target',
        columns='model',
        values='val_r2'
    )
    
    print("\nValidation R² by Target and Model:")
    display(pivot.round(3))
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    pivot.plot(kind='bar', ax=ax)
    ax.set_ylabel('Validation R²')
    ax.set_title('Model Performance Comparison')
    ax.legend(title='Model')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'baseline_comparison.png', dpi=150)
    plt.show()

## 5. Feature Importance Analysis

In [ ]:
# Get feature importance from best model
if models and available_targets:
    target = available_targets[0]  # Use first target
    rf_model = models.get((target, 'RandomForest'))
    
    if rf_model is not None:
        importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': rf_model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        # Top 20 features
        fig, ax = plt.subplots(figsize=(10, 8))
        importance.head(20).plot(kind='barh', x='feature', y='importance', ax=ax)
        ax.set_xlabel('Importance')
        ax.set_title(f'Top 20 Features for {target} (Random Forest)')
        ax.invert_yaxis()
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / f'feature_importance_{target}.png', dpi=150)
        plt.show()
        
        print("\nTop 10 features:")
        display(importance.head(10))

## 6. Ensemble Performance

In [ ]:
# Calculate ensemble predictions
if models and train_df is not None:
    ensemble_results = []
    
    for target in available_targets:
        y_val = val_df[target]
        val_mask = ~y_val.isna()
        
        if val_mask.sum() < 10:
            continue
        
        # Get predictions from each model
        ensemble_pred = np.zeros(val_mask.sum())
        
        for model_name, config in MODEL_CONFIGS.items():
            model = models.get((target, model_name))
            if model is not None:
                pred = model.predict(X_val[val_mask])
                ensemble_pred += config['weight'] * pred
        
        # Calculate ensemble metrics
        ensemble_r2 = r2_score(y_val[val_mask], ensemble_pred)
        ensemble_rmse = np.sqrt(mean_squared_error(y_val[val_mask], ensemble_pred))
        
        ensemble_results.append({
            'target': target,
            'ensemble_r2': ensemble_r2,
            'ensemble_rmse': ensemble_rmse
        })
        
        print(f"{target}: Ensemble R²={ensemble_r2:.3f}, RMSE={ensemble_rmse:.3f}")
    
    ensemble_df = pd.DataFrame(ensemble_results)
    print("\nEnsemble improves over individual models by averaging predictions.")

## 7. Save Results

In [ ]:
# Save results
if results:
    results_df.to_csv(RESULTS_DIR / 'baseline_results.csv', index=False)
    
    # Summary
    summary = {
        'models_trained': list(MODEL_CONFIGS.keys()),
        'targets': available_targets,
        'best_model_per_target': {},
        'baseline_metrics': {}
    }
    
    for target in available_targets:
        target_results = results_df[results_df['target'] == target]
        if len(target_results) > 0:
            best_idx = target_results['val_r2'].idxmax()
            best_row = target_results.loc[best_idx]
            summary['best_model_per_target'][target] = best_row['model']
            summary['baseline_metrics'][target] = {
                'val_r2': float(best_row['val_r2']),
                'val_rmse': float(best_row['val_rmse'])
            }
    
    with open(RESULTS_DIR / 'baseline_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"Results saved to {RESULTS_DIR}")

## 8. Key Takeaways for Transformer Development

Based on baseline analysis:

1. **Performance targets**: Transformer should exceed these R² values
2. **Important features**: Focus temporal encoding on high-importance weather features
3. **Challenging targets**: May need specialized heads or auxiliary tasks
4. **Ensemble advantage**: Transformer should match or beat weighted ensemble